## 4D Objects-By-Change - Aggregating objects

Due to noise in the time series, the extraction of *4D objects-by-change* (4D-OBCs; [Anders et al., 2020](#References)) may over-segment a change process, so that several 4D-OBCs describe the same change. This notebook explains how such objects are merged into one object ([Ulm et al., 2025](#References)). It builds on the [4D-OBC analysis notebook](4dobc-analysis.ipynb) and the [seed detection notebook](4dobc-seeds.ipynb).

Implemented in: `py4dgeo.SpatiotemporalAnalysis.aggregate_4dobcs`

**Implemented by**  
Robin Scharnagl (Technical University of Munich)

**Author(s) of the method**
M. Ulm, M. Elias, A. Eltner, E. Lotsari, [K. Anders](https://www.professoren.tum.de/anders-katharina)

**Original publication of the method**
Ulm, M., Elias, M., Eltner, A., Lotsari, E., & Anders, K. (2025). Automated change detection in photogrammetric 4D point clouds – transferability and extension of 4D objects-by-change for monitoring riverbank dynamics using low-cost cameras. Applied Geomatics. doi: [10.1007/s12518-025-00623-9](https://doi.org/10.1007/s12518-025-00623-9).

### Method description

- **Theoretical foundation**: Due to uncorrelated noise, the extraction of 4D-OBCs tends to oversegment, which leads to multiple 4D-OBCs describing one process. This function merges 4D-OBCs that overlap in time and space, to create just one object.

- **Algorithm**: The algorithm first checks the change direction of each object. It then calculates the spatial and temporal overlap of the OBCs. If the OBC's changes point in the same direction, and their temporal and spatial overlap exceeds a user-defined threshold, their link is stored. In the last step, these get fused to one single 4D-OBC

- **Key parameters**: The parameters that the user can define are the following:
    - `time_threshold` (float, 0 - 1): threshold for temporal overlap, 1: complete overlap, 0: no overlap
    - `spatial_threshold`(float, 0 - 1): thershold for spatial overlap, 1: complete overlap, 0: no overlap


In [ ]:
import py4dgeo
import os
from datetime import datetime
import pooch

We use a time series of hourly terrestrial laser scanning point clouds of the Schneeferner glacier (Zugspitze, Germany), which is available on Zenodo. The point clouds are downloaded once and cached locally. The acquisition time of each epoch is encoded in the file name, which we use to sort the files:

In [ ]:
# Download the data from Zenodo and set the path to the point cloud folder
p = pooch.Pooch(base_url="doi:10.5281/zenodo.10003574/", path=pooch.os_cache("py4dgeo"))
p.load_registry_from_doi()
p.fetch("module3.zip", processor=pooch.Unzip(), progressbar=True)
pc_dir = p.path / "module3.zip.unzip" / "module3" / "schneeferner" / "pointclouds"


def parse_timestamp(fname):
    ts_str = "_".join(fname.split(".")[0].split("_")[1:])  # YYMMDD_hhmmss
    return datetime.strptime(ts_str, "%y%m%d_%H%M%S")


# Sort the point cloud files by their acquisition time
pc_files = sorted(
    (f for f in os.listdir(pc_dir) if f.endswith(".laz")),
    key=parse_timestamp,
)
timestamps = [parse_timestamp(f) for f in pc_files]
print(f"Found {len(pc_files)} epochs from {timestamps[0]} to {timestamps[-1]}")

Next, we create the `SpatiotemporalAnalysis` object as described in the [notebook on creating analysis files](4dobc-creation.ipynb). The first epoch serves as reference epoch and all of its points are used as core points. M3C2 distances are then computed for all subsequent epochs. As all points of the reference epoch are used as core points, this step may take several minutes:

In [ ]:
analysis = py4dgeo.SpatiotemporalAnalysis(str(pc_dir / "schneeferner.zip"), force=True)

# Read the reference epoch and set its timestamp
reference_epoch = py4dgeo.read_from_las(str(pc_dir / pc_files[0]))
reference_epoch.timestamp = timestamps[0]

# Set the reference epoch and use all of its points as core points
analysis.reference_epoch = reference_epoch
analysis.corepoints = reference_epoch.cloud

# Compute M3C2 distances for all subsequent epochs
analysis.m3c2 = py4dgeo.M3C2(cyl_radius=2.0, normal_radii=[2.0])
epochs_to_add = []
for fname, ts in zip(pc_files[1:], timestamps[1:]):
    epoch = py4dgeo.read_from_las(str(pc_dir / fname))
    epoch.timestamp = ts
    epochs_to_add.append(epoch)

analysis.add_epochs(*epochs_to_add)

Next, we extract the 4D-OBCs that are to be aggregated. We use the `linear_rdp` seed detection method, which is explained together with the other seed detection methods in the [seed detection notebook](4dobc-seeds.ipynb). The region growing is performed for all seeds, so its runtime increases with the number of seeds. To speed it up, you can increase `seed_subsampling`.

In [ ]:
algo = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=2.0,
    seed_subsampling=50,
    height_threshold=0.5,
    method="linear_rdp",
)

objects = algo.run(analysis)
print(f"The segmentation extracted {len(objects)} 4D objects-by-change.")

Now we aggregate the extracted 4D-OBCs. `aggregate_4dobcs` uses the objects stored in the analysis object and returns a list of merged objects. Objects that are not linked to any other object are returned as merged objects that consist of a single 4D-OBC:

In [ ]:
merged = analysis.aggregate_4dobcs(temporal_threshold=0.1, spatial_threshold=0.3)
print(f"{len(objects)} 4D-OBCs were aggregated into {len(merged)} objects.")

We select the merged object that combines the largest number of 4D-OBCs and inspect it:

In [ ]:
largest = max(merged, key=lambda m: len(m.obj_4dobc))
print(f"Merged 4D-OBCs: {largest.obj_4dobc}")
print(f"Start epoch: {largest.start_epoch}")
print(f"End epoch: {largest.end_epoch}")
print(f"Number of core points: {len(largest.indices)}")

The plot of a merged object shows the time series of all its core points (left), colored by their mean distance within the time interval of the object, a histogram of these mean distances (center), and the location of the core points (right):

In [ ]:
largest.plot()

### References

* Anders, K., Winiwarter, L., Lindenbergh, R., Williams, J. G., Vos, S. E., & Höfle, B. (2020). 4D objects-by-change: Spatiotemporal segmentation of geomorphic surface change from LiDAR time series. ISPRS Journal of Photogrammetry and Remote Sensing, 159, pp. 352-363. doi: [10.1016/j.isprsjprs.2019.11.025](https://doi.org/10.1016/j.isprsjprs.2019.11.025).
* Ulm, M., Elias, M., Eltner, A., Lotsari, E., & Anders, K. (2025). Automated change detection in photogrammetric 4D point clouds – transferability and extension of 4D objects-by-change for monitoring riverbank dynamics using low-cost cameras. Applied Geomatics. doi: [10.1007/s12518-025-00623-9](https://doi.org/10.1007/s12518-025-00623-9).